# 10 · IA3 — learned volume knobs on the model's internals

**Recap (09):** attention builds **Key** and **Value** vectors, and there's also a
feed-forward (FFN) block with its own internal activations. **IA3** ("Infused Adapter
by Inhibiting and Amplifying Inner activations") trains a tiny set of **multipliers**
that rescale those vectors — feature by feature.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## The mechanism: multiply, don't add

LoRA *added* a low-rank change; bias *shifted*; IA3 **rescales**. For a vector of
activations, IA3 learns one multiplier per feature and multiplies element-wise —
turning each feature's "volume" up (>1) or down (<1).

Crucially the multipliers **start at 1**, so at the beginning IA3 is the identity
(model unchanged), then learns away from 1.

In [ ]:
v = rng.normal(size=4)             # e.g. the Value vector inside one attention head

scale = np.ones(4)                 # IA3 multipliers START at 1
print("at init (scale=1):", (v * scale).round(3), " identical?", np.allclose(v, v * scale))

scale = np.array([1.5, 0.2, 1.0, 0.8])   # after learning: amplify/inhibit per feature
print("after learning  :", (v * scale).round(3))

IA3 applies these learned multipliers in three places: the **Keys**, the **Values**,
and the **FFN** hidden activations. The parameter count is just the size of those
vectors. Note Qwen2.5-1.5B uses **grouped-query attention**, so K and V are only
**256-wide** (not the full 1536) — we cover that in notebook 12.

In [ ]:
kv_dim = 256        # K and V width under Qwen's grouped-query attention (2 kv-heads x 128)
d_ffn  = 8960       # FFN hidden activations
layers = 28         # Qwen2.5-1.5B has 28 layers

per_layer = kv_dim + kv_dim + d_ffn     # one multiplier per feature in K, V, FFN
total = per_layer * layers
print(f"IA3 trainable per layer : {per_layer:,}")
print(f"IA3 trainable total     : {total:,}  ({total/1.54e9:.3%} of the model)")

## Wait — if IA3 *multiplies*, why is it in the *additive* family?

Good catch, and it trips a lot of people. The PEFT family names answer a **different
question** than the operation does:

- **Operation** — *what arithmetic does the knob perform?* IA3 **multiplies**.
- **Family** — *where do the trainable parameters come from?* IA3 **adds brand-new**
  vectors that weren't in the pretrained model.

The taxonomy's **"additive" means "introduces new parameters,"** not "uses addition."
So IA3 is **multiplicative in operation** but **additive in parameters** — both true,
about different things.

The giveaway is the contrast with BitFit:

| method | family | new params added? | operation it performs |
|---|---|---|---|
| BitFit | **selective** | **no** (trains existing biases) | **addition** (a bias offset) |
| IA3 | **additive** | **yes** (new scaling vectors) | **multiplication** |
| LoRA | reparameterization | yes (new `B`, `A`) | add a low-rank update |

Notice the top two are almost **backwards**: BitFit's *operation* is addition yet it's
**selective** (no new params); IA3's *operation* is multiplication yet it's **additive**
(new params). The family label tracks **parameter provenance**, not the math.

*(Footnote: these survey categories — additive / selective / reparameterization —
vary a little between papers. A useful map, not a law.)*

## Recap

- **IA3** trains learned **multipliers** that rescale the **K**, **V**, and **FFN**
  activations element-wise.
- Multipliers **init at 1** → starts as the identity, like LoRA's zero-init.
- It's **multiplicative in operation** but lives in the **additive family** because it
  introduces **new** parameters — tiny in count, limited to amplifying/suppressing
  existing features rather than remixing them.

**BAM!** Next: **`11 — prompt tuning`.**